In [0]:
people_data = spark.read.option('multiline', False) \
    .json('/FileStore/shared_uploads/harrybwatson@protonmail.com/people.json')

In [0]:
display(people_data)

age,city,name
30,NewYork,John
34,Chicago,James
28,Boston,Robert
22,Seattle,Peter
32,Houston,Anna


In [0]:
display(people_data.filter(people_data.age >= 30))

age,city,name
30,NewYork,John
34,Chicago,James
32,Houston,Anna


The multiline option telling spark that it will span multiple line.

In [0]:
iris_data = spark.read.option('multiline', True) \
    .json('/FileStore/shared_uploads/harrybwatson@protonmail.com/iris.json')

In [0]:
display(iris_data)

petalLength,petalWidth,sepalLength,sepalWidth,species
1.4,0.2,5.1,3.5,setosa
1.4,0.2,4.9,3.0,setosa
1.3,0.2,4.7,3.2,setosa
1.5,0.2,4.6,3.1,setosa
1.4,0.2,5.0,3.6,setosa
1.7,0.4,5.4,3.9,setosa
1.4,0.3,4.6,3.4,setosa
1.5,0.2,5.0,3.4,setosa
1.4,0.2,4.4,2.9,setosa
1.5,0.1,4.9,3.1,setosa


In [0]:
display(iris_data.select("species").distinct())

species
virginica
versicolor
setosa


`PERMISSIVE` mode that allows us to deal with corrupt records during parsing.

In [0]:
employee_data = spark.read.option('multiline', True) \
    .option("mode", "PERMISSIVE") \
    .json('/FileStore/shared_uploads/harrybwatson@protonmail.com/employees.json')

In [0]:
display(employee_data)

address,contact,gender,id,name,salary
"List(Baltimore, MD)","List(List(watson@commerce.gov, 650-333-3456), List(emily@gmail.com, 238-111-7689))",Female,1,Emily Watson,54000.0
"List(Barton, TN)","List(List(johnsmith@yahoo.com, 425-231-8754))",Male,2,John Smith,67000.0
"List(Salt Lake City, UT)","List(List(peter@radio.us, null), List(peterjones@yahoo.com, 425-213-0987))",Male,3,Peter Jones,45000.0
"List(Seattle, WA)","List(List(nina@hotmail.com, 813-190-3628), List(ninajames@hotmail.com, 813-456-6509))",Female,4,Nina James,95500.0


Extract individual

In [0]:
display(employee_data.select("name", "salary", "address", "contact"))

name,salary,address,contact
Emily Watson,54000.0,"List(Baltimore, MD)","List(List(watson@commerce.gov, 650-333-3456), List(emily@gmail.com, 238-111-7689))"
John Smith,67000.0,"List(Barton, TN)","List(List(johnsmith@yahoo.com, 425-231-8754))"
Peter Jones,45000.0,"List(Salt Lake City, UT)","List(List(peter@radio.us, null), List(peterjones@yahoo.com, 425-213-0987))"
Nina James,95500.0,"List(Seattle, WA)","List(List(nina@hotmail.com, 813-190-3628), List(ninajames@hotmail.com, 813-456-6509))"


Extract individual fields from a complex data type

In [0]:
display(employee_data.select("name", "salary", "address.city", "address.state"))

name,salary,city,state
Emily Watson,54000.0,Baltimore,MD
John Smith,67000.0,Barton,TN
Peter Jones,45000.0,Salt Lake City,UT
Nina James,95500.0,Seattle,WA


In [0]:
display(employee_data.select("name", "salary", "contact.email", "contact.phone"))

name,salary,email,phone
Emily Watson,54000.0,"List(watson@commerce.gov, emily@gmail.com)","List(650-333-3456, 238-111-7689)"
John Smith,67000.0,List(johnsmith@yahoo.com),List(425-231-8754)
Peter Jones,45000.0,"List(peter@radio.us, peterjones@yahoo.com)","List(null, 425-213-0987)"
Nina James,95500.0,"List(nina@hotmail.com, ninajames@hotmail.com)","List(813-190-3628, 813-456-6509)"


In [0]:
from pyspark.sql import functions as F

In [0]:
display(employee_data.select(F.col("contact.email")
                             .getItem(0)
                             .alias("email_address")))

email_address
watson@commerce.gov
johnsmith@yahoo.com
peter@radio.us
nina@hotmail.com


In [0]:
display(employee_data.select("name",
                             F.col("contact.email").getItem(0).alias("email_address"),
                             F.col("contact.phone").getItem(1).alias("phone_number")))

name,email_address,phone_number
Emily Watson,watson@commerce.gov,238-111-7689
John Smith,johnsmith@yahoo.com,null
Peter Jones,peter@radio.us,425-213-0987
Nina James,nina@hotmail.com,813-456-6509
